# Module 1: Foundations

Build a minimal Strands agent, run it, and inspect every step of the **agentic loop** in action.

![Strands agent loop: Input and Context flows into Reasoning LLM, then Tool Selection, then Tool Execution, which loops back to Reasoning until done, then Response](./agent-loop.png)

The agentic loop is the core of every Strands agent:

1. **Input & Context**: the user prompt enters the loop
2. **Reasoning (LLM)**: the model decides what to do next
3. **Tool Selection**: if it needs data, the model picks a tool
4. **Tool Execution**: your Python function runs, feeding its result back into step 2
5. **Response**: the loop exits once the model has enough information

You write the tools. Strands runs the loop.

**Prerequisites:** Python 3.10+, AWS credentials with Amazon Bedrock access.

In [ ]:
!uv pip install -q -r requirements.txt

---

## Part 1: Define a Tool

Tools are plain Python functions decorated with `@tool`. The model reads the **docstring** to decide when and how to call them. The docstring is the routing logic, not code.

Write docstrings for the model, not for other developers.

In [ ]:
from strands import Agent, tool

@tool
def get_market_data(segment: str) -> str:
    """Get subscription market data for a customer segment.

    Args:
        segment: The customer segment to look up (e.g. 'premium', 'standard', 'enterprise')
    """
    data = {
        "premium": {"willingness_to_pay": "68%", "avg_monthly": "$22", "churn_rate": "8%"},
        "standard": {"willingness_to_pay": "41%", "avg_monthly": "$14", "churn_rate": "18%"},
        "enterprise": {"willingness_to_pay": "85%", "avg_monthly": "$45", "churn_rate": "4%"},
    }
    row = data.get(segment.lower(), {"error": f"no data for segment '{segment}'"})
    return str(row)

print("Tool defined. Docstring the model will read:")
print(get_market_data.__doc__)

---

## Part 2: Create and Run the Agent

Wire the tool into an `Agent` with a system prompt and call it.

In [ ]:
agent = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

result = agent("What percentage of premium segment customers are willing to pay for a subscription tier?")

---

## Part 3: Inspect the Agent Loop

Every step is stored in `agent.messages`. Let's see what happened inside the loop.

In [ ]:
import json as _json

print(f"Total messages: {len(agent.messages)}")
print("=" * 60)

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    content = msg.get("content", [])

    if role == "user":
        text = content if isinstance(content, str) else next(
            (b.get("text", "") for b in content if isinstance(b, dict) and "text" in b), "")
        print(f"[{i}] USER: {text[:80]}")

    elif role == "assistant":
        blocks = content if isinstance(content, list) else [{"text": str(content)}]
        for block in blocks:
            if "text" in block and block["text"]:
                print(f"[{i}] LLM RESPONSE: {block['text'][:80]}")
            elif "toolUse" in block:
                tu = block["toolUse"]
                print(f"[{i}] TOOL CALL: {tu['name']}({_json.dumps(tu.get('input', {}))})")

    elif role == "tool":
        blocks = content if isinstance(content, list) else [content]
        for block in blocks:
            result_content = block.get("content", []) if isinstance(block, dict) else []
            for item in result_content:
                if "text" in item:
                    print(f"[{i}] TOOL RESULT: {item['text'][:80]}")

---

## Part 4: Built-in Observability

Strands gives you two zero-config observability tools. No setup needed.

**Lens 1: `result.metrics`:** token usage and cycle count on every call.  
**Lens 2: DEBUG logging:** shows every internal step in real time.

In [ ]:
# Lens 1: metrics on every AgentResult
summary = result.metrics.get_summary()
print("Loop metrics:")
print(f"  LLM calls (cycles):  {summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {summary.get('input_tokens', 'n/a')}")
print(f"  Output tokens:       {summary.get('output_tokens', 'n/a')}")
print(f"  Total tokens:        {summary.get('total_tokens', 'n/a')}")

In [ ]:
import logging

# Lens 2: DEBUG logging -- shows every reasoning step, tool call, and token event
# Enable for one call, then disable so the next cells stay clean
logging.getLogger("strands").setLevel(logging.DEBUG)
logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")

agent2 = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

_ = agent2("What is the churn rate for enterprise segment customers?")

# Disable after this cell
logging.getLogger("strands").setLevel(logging.WARNING)
print("\nDEBUG logging disabled for subsequent cells.")

---

## Part 5: Multi-turn Conversation

The agent keeps its conversation history across calls. Each `agent(...)` call adds to the same context window. The agent remembers what happened in previous turns.

In [ ]:
# Multi-turn: the agent remembers previous turns
agent3 = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

agent3("What is the willingness to pay for the premium segment?")
agent3("And for the standard segment?")
response = agent3("Which segment has the lower churn rate between those two?")

print(f"\nConversation length: {len(agent3.messages)} messages")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` | Python function the model can call; docstring is the routing logic |
| `Agent(tools=[], system_prompt=...)` | Assembles model + tools + instructions |
| `agent.messages` | Full loop history: user turns, LLM responses, tool calls, tool results |
| `result.metrics.get_summary()` | Token usage and cycle count on every call |
| DEBUG logging | Real-time view of every loop step, no extra config |
| Multi-turn | Each `agent(...)` call extends the same context window |

---

## What is next

In **Module 2: Single Agent**, you will use these same building blocks with three real business intelligence tools to build a complete Decision Intelligence agent for the NovaCart brief.